In [1]:
import datasets

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ds = datasets.load_dataset("qwedsacf/competition_math",split="train")

ds = ds.remove_columns("level")
ds = ds.remove_columns("type")

ds = ds.add_column("input",[""]*len(ds))

ds = ds.rename_column("solution","output")
ds = ds.rename_column("problem","instruction")

In [3]:
def last_boxed_only_string(example):
    string = example['output']
    idx = string.rfind("\\boxed")
    if idx < 0:
        idx = string.rfind("\\fbox")
        left_idx = idx + 6
        if idx < 0:
            return None
    else:
        left_idx=idx+7

    i = idx
    right_brace_idx = None
    num_left_braces_open = 0
    while i < len(string):
        if string[i] == "{":
            num_left_braces_open += 1
        if string[i] == "}":
            num_left_braces_open -= 1
            if num_left_braces_open == 0:
                right_brace_idx = i
                break
        i += 1
    
    if right_brace_idx == None:
        retval = None
    else:
        #retval = string[idx:right_brace_idx + 1]
        retval = string[left_idx:right_brace_idx]
    
    example['answer'] = retval
    return example

In [4]:
ds = ds.add_column("answer",[""]*len(ds))



In [5]:
print(ds)

Dataset({
    features: ['instruction', 'output', 'input', 'answer'],
    num_rows: 12500
})


In [6]:
ds = ds.map(last_boxed_only_string)#,num_proc=24)

Map: 100%|██████████████████████████████████████████████████████████| 12500/12500 [00:00<00:00, 23299.89 examples/s]


In [11]:
print(ds[888])

{'instruction': 'What is the largest value of $x$, if $\\frac{x}{5} + \\frac{1}{5x} = \\frac{1}{2}$?', 'output': 'We multiply both sides of the equation by $10x$ to clear the fractions, leaving us with $2x^2 + 2 = 5x$. Rearranging the terms, we have $2x^2 - 5x + 2 = 0$. We can now solve for $x$ by factoring: $(2x - 1)(x - 2) = 0$. We could also use the quadratic formula:  $$x = \\frac{5 \\pm \\sqrt{(-5)^2 - 4(2)(2)}}{4}.$$Either way, we find that $x = 1/2$ or $x = 2$. Since we want the largest value of $x$, our answer is $\\boxed 2$.', 'input': '', 'answer': None}


In [8]:
print(int(ds[9]['answer']))

ValueError: invalid literal for int() with base 10: '\\$42409'

In [9]:
ds.to_json("test.json")

Creating json from Arrow format: 100%|█████████████████████████████████████████████| 13/13 [00:00<00:00, 135.39ba/s]


10404342